In [ ]:
# ============================================================================
# STAGE 3 -- DAG ENSEMBLING (hybrid node-merging, bundle-preserving)
#
# STANDALONE single cell. Reads Stage 2.1 *_eval_dags_with_answer.jsonl,
# groups DAGs by (dataset, question_id), and fuses each question's valid
# single-trace DAGs into ONE ensemble DAG (paper Sec. 3.3 + Answer-node ext).
#
# Uses only DAGs with parse_ok=True AND a non-null answer_node_id.
# Hybrid merge: embedding prune (tau') -> Qwen2.5-32B-Instruct judge.
# Answer nodes ensembled too, by normalized-text exact match.
# All weights cached under /work/hdd/bfrc.
#
# Set HF_*/VLLM_* env vars in the shell BEFORE launching Jupyter (see Stage 2).
# ============================================================================

# ----------------------------------------------------------------------------
# CACHE REDIRECTION + SANITY CHECK (same belt-and-suspenders as Stage 2)
# ----------------------------------------------------------------------------
import os
os.environ["VLLM_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_MOE_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_DEEP_GEMM_WARMUP"] = "skip"

HF_CACHE_ROOT = "/work/hdd/bfrc"
os.environ["HF_HOME"]                 = f"{HF_CACHE_ROOT}/hf"
os.environ["HF_HUB_CACHE"]            = f"{HF_CACHE_ROOT}/hf/hub"
os.environ["TRANSFORMERS_CACHE"]      = f"{HF_CACHE_ROOT}/hf"
os.environ["HF_DATASETS_CACHE"]       = f"{HF_CACHE_ROOT}/hf/datasets"
os.environ["VLLM_CACHE_ROOT"]         = f"{HF_CACHE_ROOT}/vllm"
os.environ["TRITON_CACHE_DIR"]        = f"{HF_CACHE_ROOT}/triton"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = f"{HF_CACHE_ROOT}/torch_inductor"
os.environ["TMPDIR"]                  = f"{HF_CACHE_ROOT}/tmp"
for p in (os.environ["HF_HOME"], os.environ["HF_HUB_CACHE"],
          os.environ["HF_DATASETS_CACHE"], os.environ["VLLM_CACHE_ROOT"],
          os.environ["TRITON_CACHE_DIR"], os.environ["TORCHINDUCTOR_CACHE_DIR"],
          os.environ["TMPDIR"]):
    os.makedirs(p, exist_ok=True)

import importlib, sys
if "huggingface_hub" in sys.modules:
    importlib.reload(sys.modules["huggingface_hub"])
    if "huggingface_hub.constants" in sys.modules:
        importlib.reload(sys.modules["huggingface_hub.constants"])
from huggingface_hub import constants as _hf_constants
_resolved_hub_cache = str(_hf_constants.HF_HUB_CACHE)
print(f"[cache check] huggingface_hub resolved HF_HUB_CACHE = {_resolved_hub_cache}")
if not _resolved_hub_cache.startswith(HF_CACHE_ROOT):
    print(f"[cache check] WARNING: HF cache is outside {HF_CACHE_ROOT}. "
          f"Restart the kernel with the env vars set BEFORE launching Jupyter "
          f"if you don't want partial downloads in ~/.cache/huggingface.")
else:
    print(f"[cache check] OK -- writing under {HF_CACHE_ROOT}")


# ----------------------------------------------------------------------------
# Imports
# ----------------------------------------------------------------------------
import gc, glob, json, re, time
from collections import defaultdict
from typing import Dict, List, Optional, Set, Tuple, FrozenSet

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from sentence_transformers import SentenceTransformer


# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
DAG_INPUT_DIR   = "./outputs_2234"
DAG_OUTPUT_DIR  = "./outputs_2234"
os.makedirs(DAG_OUTPUT_DIR, exist_ok=True)

# Read the Stage 2.1 outputs (the ones with Answer nodes appended).
INPUT_GLOB = os.path.join(DAG_INPUT_DIR, "*_eval_dags_with_answer.jsonl")

# Where vLLM should download weights -- the bulletproof override.
VLLM_DOWNLOAD_DIR = f"{HF_CACHE_ROOT}/hf/hub"
os.makedirs(VLLM_DOWNLOAD_DIR, exist_ok=True)

# Embedding model for the prune step.
EMBED_MODEL  = "sentence-transformers/all-mpnet-base-v2"
EMBED_CACHE  = f"{HF_CACHE_ROOT}/hf/sentence_transformers"
TAU_PRIME    = 0.55   # permissive prune threshold (paper: tau' < tau).
                      # Low on purpose -- the LLM judge is the actual arbiter.

# LLM judge -- same Qwen2.5-32B-Instruct used by Stage 2 as the extractor.
JUDGE_MODEL    = "Qwen/Qwen2.5-32B-Instruct"
JUDGE_TEMP     = 0.0    # deterministic YES / NO
JUDGE_TOP_P    = 1.0
JUDGE_MAX_TOK  = 8      # we only need one token
JUDGE_MAX_LEN  = 4096
JUDGE_GPU_UTIL = 0.90
JUDGE_TP       = 1      # tensor_parallel_size -- bump for multi-GPU

# Output: one ensemble DAG per (dataset, question_id).
OUTPUT_SUFFIX  = "_ensemble_dags.jsonl"


# ============================================================================
# 1. LOAD + FILTER  ->  { (dataset, question_id): [dag_record, ...] }
# ============================================================================
def load_valid_dags() -> Dict[Tuple[str, str], List[Dict]]:
    grouped: Dict[Tuple[str, str], List[Dict]] = defaultdict(list)
    paths = sorted(glob.glob(INPUT_GLOB))
    if not paths:
        raise RuntimeError(f"No input JSONLs at {INPUT_GLOB}. Did Stage 2.1 run?")

    print(f"\n[stage 3] reading {len(paths)} input JSONLs:")
    total = kept = drop_parse = drop_no_answer = drop_failed = 0
    for p in paths:
        print(f"  {p}")
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                total += 1
                if not rec.get("parse_ok"):
                    drop_parse += 1
                    continue
                if rec.get("answer_node_id") is None:
                    drop_no_answer += 1
                    continue
                if rec.get("answer_from_failed_dag"):
                    # Defensive -- these also have parse_ok=False so the check
                    # above already caught them, but be explicit.
                    drop_failed += 1
                    continue
                kept += 1
                grouped[(rec.get("dataset", ""), rec.get("question_id", ""))].append(rec)

    print(f"\n[stage 3] filter: total={total} kept={kept} "
          f"dropped_parse_fail={drop_parse} "
          f"dropped_no_answer_node={drop_no_answer} "
          f"dropped_answer_from_failed={drop_failed}")
    print(f"[stage 3] {len(grouped)} unique (dataset, question_id) groups")
    sizes = [len(v) for v in grouped.values()]
    if sizes:
        print(f"[stage 3] traces-per-question: min={min(sizes)} max={max(sizes)} "
              f"mean={np.mean(sizes):.2f} median={int(np.median(sizes))}")
    return grouped


# ============================================================================
# 2. GLOBAL ID NAMESPACING
#    trace key  : "{model}::{sample_idx}"
#    global id  : "{trace_key}::{local_id}"
# ============================================================================
def trace_key(rec: Dict) -> str:
    return f"{rec['model']}::{int(rec['sample_idx'])}"


def globalize_dag(rec: Dict) -> Dict:
    """Returns a copy of rec['dag'] with all ids namespaced to this trace,
    bundles rewritten to global ids, and a source_trace tag on every node.
    The original record is not mutated."""
    tk = trace_key(rec)
    def G(nid: str) -> str:
        return f"{tk}::{nid}"

    nodes_out = []
    for n in rec["dag"]["nodes"]:
        nodes_out.append({
            "id":           G(n["id"]),
            "type":         n["type"],
            "text":         n.get("text", ""),
            "support":      [[G(x) for x in b] for b in n.get("support", [])],
            "defeats":      [[G(x) for x in b] for b in n.get("defeats", [])],
            "source_trace": tk,
            "original_id":  n["id"],
        })
    return {
        "trace_key":             tk,
        "model":                 rec["model"],
        "sample_idx":            int(rec["sample_idx"]),
        "predicted_label":       rec.get("predicted_label", ""),
        "gold_label":            rec.get("gold_label", ""),
        "answer_node_id_global": G(rec["answer_node_id"]),
        "nodes":                 nodes_out,
    }


# ============================================================================
# 3. CANDIDATE PAIRS  +  4. HYBRID NODE MERGING
# ============================================================================
def normalize_answer_text(s: str) -> str:
    """Verdict-merge path: collapse case / whitespace / punctuation."""
    s = (s or "").strip().lower()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s


class UnionFind:
    def __init__(self, items):
        self.parent = {x: x for x in items}
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


JUDGE_SYSTEM = (
    "You are a strict semantic equivalence judge for reasoning-graph nodes. "
    "Two nodes should merge ONLY IF they express the SAME underlying claim, "
    "fact, or inference, regardless of wording. Lexical overlap is NOT enough. "
    "If they cite different statutes, parties, events, rules, or draw "
    "different inferences, they are DIFFERENT. "
    "Answer with a single token: YES or NO."
)

def build_judge_prompt(ntype: str, text_a: str, text_b: str,
                       question_blob: str) -> str:
    return f"""Decide whether these two reasoning-graph nodes should be merged into one.

Both are of type "{ntype}" (already verified to match).

QUESTION CONTEXT (helps interpret references like "the hypothesis"):
{question_blob}

NODE A: {text_a}
NODE B: {text_b}

Should A and B be merged because they assert the SAME underlying
claim / fact / inference?

Reply with exactly one token: YES or NO."""


def question_blob_from_rec(rec: Dict) -> str:
    """Stage 2 did not persist raw question fields into the DAG JSONL, so the
    judge gets a compact identifier plus the dataset. That is enough to
    disambiguate intra-question references in practice."""
    return f"[dataset={rec.get('dataset','')}, question_id={rec.get('question_id','')}]"


def build_candidate_pairs(globalized: List[Dict], embedder: SentenceTransformer
                          ) -> Tuple[List[Tuple[str, str]], Dict[str, Dict]]:
    """Returns (judge_pairs, id_to_node).
       judge_pairs are SAME-TYPE, CROSS-TRACE pairs that survived the
       embedding prune and need LLM verification. Answer-Answer pairs are
       handled separately by merge_answer_nodes()."""
    id_to_node: Dict[str, Dict] = {}
    nodes_by_type: Dict[str, List[Dict]] = defaultdict(list)
    for g in globalized:
        for n in g["nodes"]:
            id_to_node[n["id"]] = n
            nodes_by_type[n["type"]].append(n)

    judge_pairs: List[Tuple[str, str]] = []
    for ntype, nodes in nodes_by_type.items():
        if ntype == "Answer":
            continue  # merged by exact normalized text, not embed/LLM
        if len(nodes) < 2:
            continue
        texts = [n["text"] for n in nodes]
        embs = embedder.encode(texts, batch_size=64, convert_to_numpy=True,
                               normalize_embeddings=True, show_progress_bar=False)
        sim = embs @ embs.T  # cosine, embeddings are L2-normalized
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                if nodes[i]["source_trace"] == nodes[j]["source_trace"]:
                    continue  # never merge within a trace
                if sim[i, j] >= TAU_PRIME:
                    judge_pairs.append((nodes[i]["id"], nodes[j]["id"]))
    return judge_pairs, id_to_node


def merge_answer_nodes(id_to_node: Dict[str, Dict]) -> List[Tuple[str, str]]:
    """Answer nodes with identical normalized text are merged. Returns a
    spanning set of pairs (each Answer linked to the first equivalent one)."""
    by_norm: Dict[str, List[str]] = defaultdict(list)
    for nid, n in id_to_node.items():
        if n["type"] == "Answer":
            by_norm[normalize_answer_text(n["text"])].append(nid)
    pairs: List[Tuple[str, str]] = []
    for ids in by_norm.values():
        for other in ids[1:]:
            pairs.append((ids[0], other))
    return pairs


# ============================================================================
# 5. BUNDLE-PRESERVING MERGE  (paper eq. 2)
# ============================================================================
def bundle_key(bundle: List[str], uf: UnionFind) -> FrozenSet[str]:
    return frozenset(uf.find(x) for x in bundle)


def merge_dags(globalized: List[Dict], uf: UnionFind) -> Dict:
    """Build the ensemble DAG given the clustering pi (encoded in uf).

    Each ensemble node:
      id            : cluster representative id
      type          : node type (shared by all members)
      text          : representative text
      member_texts  : every (trace, original_id, text) that mapped in
      source_traces : set of traces contributing to the cluster
      n_attest      : node-level attestation = |source_traces|
      support/defeats : bundles, each {members, attestations, n_attest}
      gate          : Atomic / And / Or (re-derived from |support|)"""
    clusters: Dict[str, Dict] = {}
    member_root: Dict[str, str] = {}

    # Pass 1 -- init clusters, collect membership.
    for g in globalized:
        for n in g["nodes"]:
            root = uf.find(n["id"])
            member_root[n["id"]] = root
            c = clusters.get(root)
            if c is None:
                c = clusters[root] = {
                    "id": root, "type": n["type"], "text": n["text"],
                    "member_texts": [], "source_traces": set(),
                    "support": {}, "defeats": {},
                }
            c["member_texts"].append({
                "trace": n["source_trace"],
                "original_id": n["original_id"],
                "text": n["text"],
            })
            c["source_traces"].add(n["source_trace"])

    # Pass 2 -- aggregate bundles through pi (eq. 2).
    for g in globalized:
        tk = g["trace_key"]
        for n in g["nodes"]:
            c = clusters[member_root[n["id"]]]
            root = c["id"]
            for kind in ("support", "defeats"):
                for bundle in n.get(kind, []):
                    if not bundle:
                        continue
                    bk = bundle_key(bundle, uf)
                    if not bk:
                        continue
                    # Drop a bundle that collapsed onto the node itself
                    # (a self-loop; can occur if a node merged with one of
                    # its own predecessors).
                    if bk == {root}:
                        continue
                    slot = c[kind].get(bk)
                    if slot is None:
                        slot = c[kind][bk] = {"members": sorted(bk),
                                              "attestations": set()}
                    slot["attestations"].add(tk)

    # Finalize: sets -> lists, sort bundles strongest-first.
    nodes_out = []
    for root, c in clusters.items():
        def finalize(d):
            out = [{"members": v["members"],
                    "attestations": sorted(v["attestations"]),
                    "n_attest": len(v["attestations"])}
                   for v in d.values()]
            out.sort(key=lambda b: (-b["n_attest"], len(b["members"])))
            return out
        nodes_out.append({
            "id": root, "type": c["type"], "text": c["text"],
            "member_texts": c["member_texts"],
            "source_traces": sorted(c["source_traces"]),
            "n_attest": len(c["source_traces"]),
            "support": finalize(c["support"]),
            "defeats": finalize(c["defeats"]),
        })

    # Re-derive gates from |support|  (paper Sec. 3.2).
    for n in nodes_out:
        ns = len(n["support"])
        n["gate"] = "Atomic" if ns == 0 else ("And" if ns == 1 else "Or")
    return {"nodes": nodes_out}


# ============================================================================
# 8. CYCLE BREAKING -- remove the weakest support bundle along each cycle
# ============================================================================
def break_cycles(ens: Dict) -> int:
    """Mutates ens in place. Returns the number of support bundles removed."""
    id_to_node = {n["id"]: n for n in ens["nodes"]}
    n_removed = 0

    def find_cycle() -> Optional[List[Tuple[str, int]]]:
        """DFS over support edges. Returns a list of (node_id, bundle_idx)
        edges forming a cycle, or None if the graph is acyclic."""
        WHITE, GRAY, BLACK = 0, 1, 2
        color = {nid: WHITE for nid in id_to_node}
        parent_edge: Dict[str, Tuple[str, int]] = {}

        for start in id_to_node:
            if color[start] != WHITE:
                continue
            color[start] = GRAY
            stack: List[Tuple[str, int, int]] = [(start, 0, 0)]
            while stack:
                node, b_idx, m_idx = stack[-1]
                bundles = id_to_node[node]["support"]
                stepped = False
                while b_idx < len(bundles):
                    members = bundles[b_idx]["members"]
                    if m_idx < len(members):
                        nxt = members[m_idx]
                        stack[-1] = (node, b_idx, m_idx + 1)
                        if nxt not in id_to_node:
                            m_idx += 1
                            continue
                        col = color.get(nxt, WHITE)
                        if col == GRAY:
                            # back edge -- node --bundle b_idx--> nxt closes a cycle
                            cyc = [(node, b_idx)]
                            cur = node
                            while cur != nxt:
                                p, pb = parent_edge[cur]
                                cyc.append((p, pb))
                                cur = p
                            return cyc
                        if col == WHITE:
                            color[nxt] = GRAY
                            parent_edge[nxt] = (node, b_idx)
                            stack.append((nxt, 0, 0))
                            stepped = True
                            break
                        m_idx += 1
                    else:
                        b_idx += 1
                        m_idx = 0
                        stack[-1] = (node, b_idx, m_idx)
                if not stepped and stack and stack[-1][0] == node \
                        and stack[-1][1] >= len(bundles):
                    color[node] = BLACK
                    stack.pop()
        return None

    while True:
        cyc = find_cycle()
        if cyc is None:
            break
        # weakest = fewest attestations; tie-break larger bundle first
        wn, wi = min(cyc, key=lambda e: (
            id_to_node[e[0]]["support"][e[1]]["n_attest"],
            -len(id_to_node[e[0]]["support"][e[1]]["members"]),
        ))
        del id_to_node[wn]["support"][wi]
        n_removed += 1

    for n in ens["nodes"]:
        ns = len(n["support"])
        n["gate"] = "Atomic" if ns == 0 else ("And" if ns == 1 else "Or")
    return n_removed


# ============================================================================
# DRIVER
# ============================================================================
def already_done_keys(out_path: str) -> Set[Tuple[str, str]]:
    done: Set[Tuple[str, str]] = set()
    if not os.path.exists(out_path):
        return done
    with open(out_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
                done.add((rec["dataset"], rec["question_id"]))
            except (json.JSONDecodeError, KeyError):
                continue
    return done


grouped = load_valid_dags()

# Group questions by dataset for per-dataset output files.
by_dataset: Dict[str, List[Tuple[str, List[Dict]]]] = defaultdict(list)
for (ds, qid), recs in grouped.items():
    by_dataset[ds].append((qid, recs))

# Resume.
work_per_dataset: Dict[str, List[Tuple[str, List[Dict]]]] = {}
for ds, items in by_dataset.items():
    out_path = os.path.join(DAG_OUTPUT_DIR, f"{ds}{OUTPUT_SUFFIX}")
    done = already_done_keys(out_path)
    pending = [(q, r) for q, r in items if (ds, q) not in done]
    work_per_dataset[ds] = pending
    print(f"  [{ds}] {len(items)} questions, "
          f"{len(items) - len(pending)} already done, {len(pending)} pending")

n_pending = sum(len(v) for v in work_per_dataset.values())


if n_pending == 0:
    print("\n[stage 3] nothing to do; all questions already ensembled.")
else:
    # ------------------------------------------------------------------
    # Load models.
    # ------------------------------------------------------------------
    print(f"\n[stage 3] loading embedder {EMBED_MODEL}")
    embedder = SentenceTransformer(EMBED_MODEL, cache_folder=EMBED_CACHE)
    if torch.cuda.is_available():
        embedder = embedder.to("cuda")

    print(f"[stage 3] loading judge {JUDGE_MODEL}")
    print(f"          weights download dir: {VLLM_DOWNLOAD_DIR}")
    tokenizer = AutoTokenizer.from_pretrained(
        JUDGE_MODEL, cache_dir=VLLM_DOWNLOAD_DIR, trust_remote_code=True)
    llm = LLM(
        model=JUDGE_MODEL,
        dtype="bfloat16",
        trust_remote_code=True,
        gpu_memory_utilization=JUDGE_GPU_UTIL,
        max_model_len=JUDGE_MAX_LEN,
        tensor_parallel_size=JUDGE_TP,
        download_dir=VLLM_DOWNLOAD_DIR,   # bulletproof: weights -> /work/hdd/bfrc
    )

    # ------------------------------------------------------------------
    # Stage A -- build candidate pairs for every question.
    # ------------------------------------------------------------------
    print(f"\n[stage 3] building candidate pairs across {n_pending} questions ...")
    t0 = time.time()
    per_question_state: List[Dict] = []
    global_judge_pairs: List[Tuple[int, Tuple[str, str]]] = []
    for ds, items in work_per_dataset.items():
        for qid, recs in items:
            globalized = [globalize_dag(r) for r in recs]
            judge_pairs, id_to_node = build_candidate_pairs(globalized, embedder)
            answer_pairs = merge_answer_nodes(id_to_node)
            qidx = len(per_question_state)
            per_question_state.append({
                "dataset": ds, "question_id": qid, "recs": recs,
                "globalized": globalized, "id_to_node": id_to_node,
                "judge_pairs": judge_pairs, "answer_pairs": answer_pairs,
                "question_blob": question_blob_from_rec(recs[0]),
            })
            for pr in judge_pairs:
                global_judge_pairs.append((qidx, pr))
    print(f"[stage 3] built pairs in {time.time()-t0:.1f}s; "
          f"{len(global_judge_pairs)} judge calls queued")

    # ------------------------------------------------------------------
    # Stage B -- one big batched judge call.
    # ------------------------------------------------------------------
    approved_per_q: Dict[int, Set[Tuple[str, str]]] = defaultdict(set)
    if global_judge_pairs:
        print(f"\n[stage 3] dispatching {len(global_judge_pairs)} judge prompts ...")
        t0 = time.time()
        prompts, sps = [], []
        for qidx, (a, b) in global_judge_pairs:
            st = per_question_state[qidx]
            na, nb = st["id_to_node"][a], st["id_to_node"][b]
            user = build_judge_prompt(na["type"], na["text"], nb["text"],
                                      st["question_blob"])
            messages = [{"role": "system", "content": JUDGE_SYSTEM},
                        {"role": "user",   "content": user}]
            prompts.append(tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False))
            sps.append(SamplingParams(temperature=JUDGE_TEMP, top_p=JUDGE_TOP_P,
                                      max_tokens=JUDGE_MAX_TOK, seed=0, n=1))
        outputs = llm.generate(prompts, sps)
        dt = time.time() - t0
        print(f"[stage 3] judge done in {dt:.1f}s "
              f"({dt/max(1,len(prompts)):.3f}s/call avg)")
        for (qidx, pair), out in zip(global_judge_pairs, outputs):
            if out.outputs[0].text.strip().upper().startswith("YES"):
                approved_per_q[qidx].add(pair)
        n_yes = sum(len(v) for v in approved_per_q.values())
        print(f"[stage 3] judge approved {n_yes}/{len(global_judge_pairs)} merges")

    # ------------------------------------------------------------------
    # Stage C -- per-question merge + cycle-break + write.
    # ------------------------------------------------------------------
    print("\n[stage 3] merging + cycle-breaking per question ...")
    files: Dict[str, "io.TextIOBase"] = {}
    n_done = n_cycles_total = 0
    try:
        for qidx, st in enumerate(per_question_state):
            ds, qid = st["dataset"], st["question_id"]
            id_to_node = st["id_to_node"]

            # Union-find -> clustering pi.
            uf = UnionFind(list(id_to_node.keys()))
            for a, b in approved_per_q.get(qidx, set()):
                uf.union(a, b)
            for a, b in st["answer_pairs"]:        # Answer nodes merged too
                uf.union(a, b)

            ensemble = merge_dags(st["globalized"], uf)
            n_rm = break_cycles(ensemble)
            n_cycles_total += n_rm

            # Identify the merged Answer-node clusters and their support.
            answer_cluster_ids = {uf.find(g["answer_node_id_global"])
                                  for g in st["globalized"]}
            ens_lookup = {n["id"]: n for n in ensemble["nodes"]}
            answer_summary = []
            for cid in answer_cluster_ids:
                n = ens_lookup.get(cid)
                if n is not None:
                    answer_summary.append({
                        "answer_cluster_id": cid,
                        "text":              n["text"],
                        "n_attest":          n["n_attest"],
                        "attesting_traces":  n["source_traces"],
                    })
            answer_summary.sort(key=lambda x: -x["n_attest"])

            out_rec = {
                "dataset":      ds,
                "question_id":  qid,
                "gold_label":   st["recs"][0].get("gold_label", ""),
                "n_source_traces": len(st["globalized"]),
                "source_traces":   [g["trace_key"] for g in st["globalized"]],
                "per_trace_predicted": {g["trace_key"]: g["predicted_label"]
                                        for g in st["globalized"]},
                "n_merge_pairs_proposed": len(st["judge_pairs"]),
                "n_merge_pairs_approved": len(approved_per_q.get(qidx, set())),
                "n_answer_merge_pairs":   len(st["answer_pairs"]),
                "n_cycles_broken":        n_rm,
                "answer_clusters":        answer_summary,
                "ensemble_dag":           ensemble,
            }
            out_path = os.path.join(DAG_OUTPUT_DIR, f"{ds}{OUTPUT_SUFFIX}")
            if out_path not in files:
                files[out_path] = open(out_path, "a", encoding="utf-8")
            files[out_path].write(json.dumps(out_rec, ensure_ascii=False) + "\n")
            files[out_path].flush()
            n_done += 1
    finally:
        for f in files.values():
            f.close()

    print(f"[stage 3] wrote {n_done} ensemble DAGs; "
          f"broke {n_cycles_total} cycle-bundles total")

    # Free GPU.
    del llm, tokenizer, embedder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ----------------------------------------------------------------------------
# Summary table.
# ----------------------------------------------------------------------------
print("\n[stage 3] final tallies:")
for p in sorted(glob.glob(os.path.join(DAG_OUTPUT_DIR, f"*{OUTPUT_SUFFIX}"))):
    n = n_nodes = n_traces = n_consensus = 0
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            n += 1
            n_nodes  += len(rec["ensemble_dag"]["nodes"])
            n_traces += rec["n_source_traces"]
            ac = rec.get("answer_clusters") or []
            if ac and ac[0]["n_attest"] > 1:
                n_consensus += 1
    if n:
        print(f"  {os.path.basename(p)}: {n} questions | "
              f"avg {n_nodes/n:.1f} nodes/ensemble | "
              f"avg {n_traces/n:.1f} traces/question | "
              f"{n_consensus}/{n} have a multi-trace consensus answer")